# Assignment 1 — NumPy: vectorised computation
No Python `for` loop over array elements is used anywhere in this notebook.
The only plain loop is in Q8, as the question requires, and it runs over a Python **list** (`.tolist()`), not over a NumPy array.

In [18]:
import time, timeit
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

np.random.seed(42)

## Q1. Create `A` (1000 × 5), normal(mean 50, std 12)

In [19]:
A = np.random.normal(loc=50, scale=12, size=(1000, 5))
print("shape :", A.shape)
print("dtype :", A.dtype)
print("memory (float64):", A.nbytes, "bytes")

A32 = A.astype(np.float32)
print("memory (float32):", A32.nbytes, "bytes")

shape : (1000, 5)
dtype : float64
memory (float64): 40000 bytes
memory (float32): 20000 bytes


## Q2. Standardise every column (one broadcast expression)

In [20]:
Z = (A - A.mean(axis=0)) / A.std(axis=0)

print("column means:", np.round(Z.mean(axis=0), 6))
print("column stds :", np.round(Z.std(axis=0), 6))

column means: [-0. -0.  0. -0. -0.]
column stds : [1. 1. 1. 1. 1.]


## Q3. `minmax(X, axis)`

In [21]:
def minmax(X, axis):
    mn = X.min(axis=axis, keepdims=True)
    mx = X.max(axis=axis, keepdims=True)
    return (X - mn) / (mx - mn)

M0 = minmax(A, axis=0)
print("axis=0 -> min per column:", M0.min(axis=0))
print("axis=0 -> max per column:", M0.max(axis=0))

M1 = minmax(A, axis=1)
print("axis=1 -> min per row (first 5):", M1.min(axis=1)[:5])
print("axis=1 -> max per row (first 5):", M1.max(axis=1)[:5])
assert np.allclose(M0.min(axis=0), 0) and np.allclose(M0.max(axis=0), 1)
assert np.allclose(M1.min(axis=1), 0) and np.allclose(M1.max(axis=1), 1)
print("all mins are 0 and all maxes are 1")

axis=0 -> min per column: [0. 0. 0. 0. 0.]
axis=0 -> max per column: [1. 1. 1. 1. 1.]
axis=1 -> min per row (first 5): [0. 0. 0. 0. 0.]
axis=1 -> max per row (first 5): [1. 1. 1. 1. 1.]
all mins are 0 and all maxes are 1


## Q4. Inject NaN into 5% of positions, fill with each column's mean

In [22]:
A_nan = A.copy()
n_nan = int(0.05 * A_nan.size)                       # exactly 5% of 5000 = 250
pos = np.random.choice(A_nan.size, size=n_nan, replace=False)
A_nan.flat[pos] = np.nan
print("NaN injected:", np.isnan(A_nan).sum(), "of", A_nan.size)
print("NaN per column:", np.isnan(A_nan).sum(axis=0))

col_means = np.nanmean(A_nan, axis=0)                # means ignoring NaN
mask = np.isnan(A_nan)                               # boolean mask of NaN positions
A_nan[mask] = np.broadcast_to(col_means, A_nan.shape)[mask]   # each NaN gets its own column's mean

print("NaN left:", np.isnan(A_nan).sum())
assert np.isnan(A_nan).sum() == 0

NaN injected: 250 of 5000
NaN per column: [54 46 53 41 56]
NaN left: 0


## Q5. Replace values beyond 3 std from the column mean with the column median

In [23]:
mu  = A_nan.mean(axis=0)
sd  = A_nan.std(axis=0)
med = np.median(A_nan, axis=0)

is_outlier = np.abs(A_nan - mu) > 3 * sd
A_clean = np.where(is_outlier, med, A_nan)           # median broadcasts per column

print("values replaced per column:", is_outlier.sum(axis=0))
print("total replaced            :", is_outlier.sum())

A = A_clean          # from here on, A is the cleaned array

values replaced per column: [5 1 5 3 5]
total replaced            : 19


## Q6. 200 × 200 Euclidean distance matrix (broadcasting only)

In [24]:
X = A[:200]                                           # (200, 5)
diff = X[:, None, :] - X[None, :, :]                  # (200, 1, 5) - (1, 200, 5) -> (200, 200, 5)
D = np.sqrt((diff ** 2).sum(axis=-1))                 # (200, 200)

print("D shape:", D.shape)
print("diagonal is all 0 :", np.all(np.diag(D) == 0))
print("D equals D.T      :", np.array_equal(D, D.T))
assert np.all(np.diag(D) == 0) and np.array_equal(D, D.T)

D shape: (200, 200)
diagonal is all 0 : True
D equals D.T      : True


## Q7. 5 nearest rows per row with `argpartition`

In [25]:
D2 = D.copy()
np.fill_diagonal(D2, np.inf)                          # a row must not be its own neighbour

idx = np.argpartition(D2, 5, axis=1)[:, :5]           # 5 smallest per row, unordered
order = np.argsort(np.take_along_axis(D2, idx, axis=1), axis=1)
nearest5 = np.take_along_axis(idx, order, axis=1)     # order those 5 by distance (cheap: 5 items)

print("shape:", nearest5.shape, "| dtype:", nearest5.dtype)
print(nearest5[:5])

# check against a full sort
check = np.argsort(D2, axis=1)[:, :5]
assert np.array_equal(np.sort(nearest5, axis=1), np.sort(check, axis=1))
assert not np.any(nearest5 == np.arange(200)[:, None])
print("matches full-sort result; no row lists itself")

shape: (200, 5) | dtype: int64
[[180 179 116 178 192]
 [114  17  31 112 197]
 [  3 158 132 145 102]
 [132   2 101   5 194]
 [160 158 143 102  43]]
matches full-sort result; no row lists itself


**Why `argpartition`:** Argpartition is used for finding the 5 smallest in O(n) per row, maenwhile a full sort costs O(n log n) and orders 195 values that we never use.

## Q8. `np.bincount` vs a plain Python loop (1,000,000 integers, 0–99)

In [26]:
data = np.random.randint(0, 100, size=1_000_000)
data.shape, data.min(), data.max()

((1000000,), np.int64(0), np.int64(99))

In [27]:
%%time
counts_np = np.bincount(data, minlength=100)

CPU times: user 3.12 ms, sys: 106 µs, total: 3.23 ms
Wall time: 2.58 ms


In [28]:
data_list = data.tolist()      # plain Python list, so the loop below is over a list, not a NumPy array

In [29]:
%%time
t0 = time.perf_counter()
counts_loop = [0] * 100
for v in data_list:
    counts_loop[v] += 1
t_loop = time.perf_counter() - t0

CPU times: user 122 ms, sys: 975 µs, total: 123 ms
Wall time: 123 ms


In [30]:
# best of 7 runs for bincount, because a single run is too short to time reliably
t_np = min(timeit.repeat(lambda: np.bincount(data, minlength=100), number=1, repeat=7))

assert np.array_equal(counts_np, np.array(counts_loop))
speedup = t_loop / t_np
print(f"Python loop : {t_loop:.4f} s")
print(f"np.bincount : {t_np:.6f} s")
print(f"Speed-up    : {speedup:.0f}x")

Python loop : 0.1225 s
np.bincount : 0.001505 s
Speed-up    : 81x


## Q9. Sliding windows of length 10 over 10,000 values

In [31]:
x = np.random.randn(10_000)
W = sliding_window_view(x, 10)
print("windows shape:", W.shape)               # (10000 - 10 + 1, 10)

win_mean = W.mean(axis=1)
pd_mean  = pd.Series(x).rolling(10).mean().to_numpy()[9:]   # first 9 entries are NaN
print("same as pandas rolling(10).mean():", np.allclose(win_mean, pd_mean))
assert np.allclose(win_mean, pd_mean)

windows shape: (9991, 10)
same as pandas rolling(10).mean(): True


## Q10. Reshape (12, 8) → (4, 3, 8), mean over the middle axis

In [32]:
B = np.arange(96).reshape(12, 8)
B3 = B.reshape(4, 3, 8)
R = B3.mean(axis=1)

print("reshaped:", B3.shape, "| result:", R.shape)
print(R)
assert np.allclose(R[0], B[0:3].mean(axis=0))

reshaped: (4, 3, 8) | result: (4, 8)
[[ 8.  9. 10. 11. 12. 13. 14. 15.]
 [32. 33. 34. 35. 36. 37. 38. 39.]
 [56. 57. 58. 59. 60. 61. 62. 63.]
 [80. 81. 82. 83. 84. 85. 86. 87.]]


**What the shape means:** the 12 rows were cut into 4 consecutive blocks of 3 rows, and the middle axis (the 3 rows in each block) was averaged away.
The result `(4, 8)` is one row of those 8 column means for each of the 4 blocks.

## Q11. Save and reload the cleaned array

In [33]:
np.save("clean.npy", A)
loaded = np.load("clean.npy")

assert np.array_equal(loaded, A)
print("loaded array is identical to A:", loaded.shape, loaded.dtype)

loaded array is identical to A: (1000, 5) float64


## README.md (written automatically from this run)

In [34]:
readme = f'''# ML Assignment 1 — NumPy

Notebook: `assignment1_numpy.ipynb` (run top to bottom with Runtime -> Restart and run all).

## Q8 speed-up
`np.bincount` vs a plain Python loop over 1,000,000 integers (0-99):

- Python loop: {t_loop:.4f} s
- np.bincount: {t_np:.6f} s
- **Speed-up: {speedup:.0f}x**

Note: the Python loop in Q8 runs over a list (`.tolist()`), as the question requires; no loop runs over a NumPy array anywhere.
'''
with open("README.md", "w") as f:
    f.write(readme)
print(readme)

# ML Assignment 1 — NumPy

Notebook: `assignment1_numpy.ipynb` (run top to bottom with Runtime -> Restart and run all).

## Q8 speed-up
`np.bincount` vs a plain Python loop over 1,000,000 integers (0-99):

- Python loop: 0.1225 s
- np.bincount: 0.001505 s
- **Speed-up: 81x**

Note: the Python loop in Q8 runs over a list (`.tolist()`), as the question requires; no loop runs over a NumPy array anywhere.

